In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

# --- 1. Load and Prepare Data ---
data = pd.read_csv("onionData.csv")

# Features (all except last 3 price columns) and Labels (last 3 columns)
X = data.drop(columns=["Min_Price", "Max_Price", "Modal_Price"])
y = data[["Min_Price", "Max_Price", "Modal_Price"]]

# Convert 'Arrival_Date' to datetime and extract features
X['Arrival_Date'] = pd.to_datetime(X['Arrival_Date'], format='%d-%m-%Y')
X['Day'] = X['Arrival_Date'].dt.day
X['Month'] = X['Arrival_Date'].dt.month
X['Year'] = X['Arrival_Date'].dt.year
X = X.drop(columns=['Arrival_Date'])

# Encode categorical variables
X = X.apply(lambda col: LabelEncoder().fit_transform(col) if col.dtypes == 'object' else col)

# --- 2. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- 3. Train Linear Regression Model (multi-output regression) ---
model = LinearRegression()
model.fit(X_train, y_train)

# --- 4. Predictions ---
y_pred = model.predict(X_test)

# --- 5. Evaluation ---
for i, col in enumerate(y.columns):
    print(f"\n=== {col} ===")
    print("R² Score:", r2_score(y_test.iloc[:, i], y_pred[:, i]))
    print("MSE:", mean_squared_error(y_test.iloc[:, i], y_pred[:, i]))

# Example prediction for the first row of test data
print("\nExample prediction vs actual:")
print("Predicted:", y_pred[0])
print("Actual   :", y_test.iloc[0].values)

accuracy=model.score(X_test,y_test)
print(f"\nAccuracy: {(accuracy*100):.2f}%")



=== Min_Price ===
R² Score: 0.216602617287613
MSE: 116319.96881409125

=== Max_Price ===
R² Score: 0.2794552714309825
MSE: 225197.13673186893

=== Modal_Price ===
R² Score: 0.3732919505420156
MSE: 144264.3921561644

Example prediction vs actual:
Predicted: [ 547.98545586 1765.51115076 1393.02317252]
Actual   : [ 500. 1221. 1150.]

Accuracy: 28.98%
